# E16 심사 대응: 표현·정규화·역할·문맥 단서 검증
**해결책 — E16의 역할별 top-4 → 합집합 → 마지막 위치 벡터 → LR을 유지합니다.**

이 노트북은 기존 p8s를 사용한 **탐색적 추가 검증**입니다. 새 외부 데이터 검증이나 사전등록 실험으로 표시하지 않습니다. 기존 결과를 읽고 새 학습처럼 표시하지 않습니다.

**실행:** Colab 새 런타임에서 위에서 아래로 실행합니다. 기본 CPU 실험에는 GPU가 필요 없습니다. 실행마다 새 폴더에 실제 학습 기록·시드별 결과·원 예측을 저장합니다. 이 배포본의 실행 출력은 비어 있습니다.

| 심사 질문 | 실행 내용 |
|---|---|
| 표현 위치와 선택 효과 | 전체 블록 표현 / 투영 전 32헤드 / 역할별 헤드 / 같은 개수 무작위 헤드 |
| 정규화 공정성 | 각 방법의 소스 내부 질문 단위 3-fold로 LR C 선택; 선택 헤드도 내부 train에서 다시 선발 |
| C 신호의 성격 | 실제 K+/K−별 C AUROC; 문맥만 읽는 word/char TF-IDF |
| 역할별 기여 | M, C, X와 두 역할 조합 및 M+C+X 비교; 입력 차원 차이를 함께 기록 |
| 소량 라벨 안정성 | 시드별 헤드 ID와 20%–80% Jaccard |
| 실제 정보원 선택 | 저장된 자체 답/문맥 답 선택의 오프라인 재생; 엄격 EM과 별칭 포함 점수를 둘 다 보고 |

소스 122개/평가 490개인 A와, 소스 122개 또는 490개/공통 평가 122개인 B를 분리합니다. 모두 시드 42–46의 반복 홀드아웃입니다. **5-fold OOF가 아닙니다.**

In [ ]:
# 1. 기존 Colab 1.6.1 결과와 혼합하지 않도록, 추가 검증 환경을 고정합니다.
import sys, subprocess, importlib.metadata, importlib.util
TARGET_SKLEARN = "1.8.0"
try:
    version = importlib.metadata.version("scikit-learn")
except importlib.metadata.PackageNotFoundError:
    version = None
if version != TARGET_SKLEARN:
    if "sklearn" in sys.modules:
        raise RuntimeError("런타임을 재시작하고 이 셀부터 실행하세요.")
    subprocess.check_call([sys.executable,"-m","pip","install","-q",f"scikit-learn=={TARGET_SKLEARN}"])
for mod,pkg in [("gdown","gdown"),("threadpoolctl","threadpoolctl"),("pandas","pandas")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("scikit-learn",importlib.metadata.version("scikit-learn"))

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json, time, warnings, itertools
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
import hashlib,shutil
try:
    IN_COLAB=importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB=False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
ROOT=Path("/content/e16_review_inputs") if IN_COLAB else Path.cwd()/"e16_review_inputs"
WORK=Path("/content/drive/MyDrive/p8s") if IN_COLAB else Path.cwd()/"e16_review_results"
ROOT.mkdir(parents=True,exist_ok=True);WORK.mkdir(parents=True,exist_ok=True)
OUT=WORK/("E16_REVIEW_"+datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid4().hex[:8])
OUT.mkdir(exist_ok=False)
CFG={"seeds":[42,43,44,45,46],"C_grid":[.01,.1,1.],"inner_folds":3,"topk":4,
     "random_reps":20,"sklearn":sklearn.__version__,"scope":"exploratory follow-up on previously inspected p8s","result_reuse":False}
(OUT/"PLAN.json").write_text(json.dumps(CFG,ensure_ascii=False,indent=2))
print("이번 새 학습 폴더:",OUT)


In [ ]:
# 2. 원래 입력 두 파일만 재사용합니다. 예측·학습 결과는 입력으로 받지 않습니다.
SOURCES={
 "p8s_behavior_v2.csv":("1RXvW6o552ZNZn1PUMRaf8CEDMIUpqQOr","fd37b0afd04972e3834ffa5f0af7f362affcfac8e20325117d37d8de15447c1c"),
 "L13_neutral_head_bank_all_rows.npz":("1GK0xi217u57Wz_kn8ycg5eSlPgjsaFNy","4ad4a040388d1e45995be7aadc7b91f6dfd1f4091b4fd452da8b4a9036b83802")}
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
    return h.hexdigest()
for name,(fid,want) in SOURCES.items():
    path=ROOT/name
    options=[path,WORK/"E14_L13_head_5fold_v1"/name,WORK/name,Path("/mnt/data")/name]
    existing=next((p for p in options if p.is_file() and sha256(p)==want),None)
    if existing is not None:
        if existing.resolve()!=path.resolve():shutil.copy2(existing,path)
    else:
        import gdown
        value=gdown.download(id=fid,output=str(path),quiet=False)
        if value is None:raise RuntimeError("원본 파일 접근 실패. 해당 파일이 있는 Drive 계정을 사용하세요.")
    assert sha256(path)==want,"원본 입력 해시 불일치: "+name
    print("INPUT VERIFIED",name)


In [ ]:
F = pd.read_csv(ROOT / 'p8s_behavior_v2.csv')
K = F.kside_corr.map({'K+': 1, 'K-': 0, 'K−': 0}).to_numpy(int)
C = F.ctx_label.map({'C+': 1, 'C-': 0, 'C−': 0}).to_numpy(int)
Y = np.where(K == 1, np.where(C == 1, 0, 1), np.where(C == 1, 3, 2))
Q, QNAME = pd.factorize(F.question, sort=True)
with np.load(ROOT / 'L13_neutral_head_bank_all_rows.npz', allow_pickle=False) as z:
    assert np.array_equal(z['rows'], np.arange(len(F)))
    H = z['HEAD_LAST'].astype('float32')
    HQ = z['HEAD_Q'].astype('float32')
    B = z['BLOCK_LAST'].astype('float32')
    CM = z['C_MASS'].astype('float32')
assert H.shape == (1224, 32, 128)
norm = np.linalg.norm(HQ, axis=2)
RAW = {'memory': norm, 'context': CM, 'conflict': np.linalg.norm(H - HQ, axis=2) / np.maximum(norm, 1e-08)}
ROLES = tuple(RAW)
FIT_LOG = []
PREDS = []
TEXT = []
CV_LOG = []
SELECTIONS = []
assert len(F)==1224 and len(QNAME)==612
assert F.groupby("question").size().eq(2).all()
assert np.isfinite(H).all() and np.isfinite(B).all()
(OUT/"ENVIRONMENT.json").write_text(json.dumps({p:importlib.metadata.version(p) for p in ["numpy","pandas","scipy","scikit-learn"]},indent=2))
(OUT/"SOURCE_HASHES.json").write_text(json.dumps({n:sha256(ROOT/n) for n in SOURCES},indent=2))


## 3. 역할별 선별과 판독
M은 질문 위치 벡터 크기, C는 문맥 어텐션 질량, X는 두 위치 벡터의 상대 변화량입니다. 목표는 각각 K, C, K≠C입니다. K≠C는 **정오 속성의 XOR**이며 의미적 모순의 직접 라벨이 아닙니다.

최종 LR에는 선택된 마지막 프롬프트 위치 벡터만 입력합니다. 이 벡터가 기존 파일명 HEAD_LAST에 해당합니다. role/full/all32는 같은 소스, 같은 LR 구현으로 학습합니다.

In [ ]:
def role_select(rows):
    tg = {'memory': K[rows], 'context': C[rows], 'conflict': K[rows] != C[rows]}
    out = {}
    for role in ROLES:
        auc = [roc_auc_score(tg[role], RAW[role][rows, h]) for h in range(32)]
        out[role] = sorted(range(32), key=lambda h: (-max(auc[h], 1 - auc[h]), h))[:4]
    return (sorted(set((h for hs in out.values() for h in hs))), out)

def matrix(rows, method, heads):
    return B[rows] if method == 'full_hidden' else H[rows].reshape(len(rows), -1) if method == 'all32' else H[rows][:, heads, :].reshape(len(rows), -1)

def fit_lr(x, y, c, tag, scale=True):
    lr = LogisticRegression(C=c, solver='lbfgs', max_iter=2000, random_state=42)
    m = make_pipeline(StandardScaler(), lr) if scale else lr
    t = time.perf_counter()
    with warnings.catch_warnings(), threadpool_limits(limits=1):
        warnings.simplefilter('error', ConvergenceWarning)
        m.fit(x, y)
    FIT_LOG.append(dict(tag=tag, c=c, rows=len(y), dim=x.shape[1], seconds=time.perf_counter() - t, reused=False))
    return m

def infer(m, x):
    with threadpool_limits(limits=1):
        return m.predict_proba(x)

def record(prob, rows, panel, budget, seed, method, rep=0):
    PREDS.append(pd.DataFrame(dict(panel=panel, budget=budget, seed=seed, method=method, rep=rep, row=rows, qid=Q[rows], true_state=Y[rows], pred_state=prob.argmax(1), K_score=prob[:, [0, 1]].sum(1), C_score=prob[:, [0, 3]].sum(1))))

## 4. 정규화 강도 선택 — 평가셋을 보지 않습니다
소스 질문 안에서 3-fold를 구성합니다. 내부 검증에 쓸 질문은 해당 fold의 헤드 선발 및 표준화에서 제외합니다. C 후보는 0.01, 0.1, 1.0이며 내부 Macro-F1 평균으로 고릅니다. 전체 소스에서 헤드를 다시 고르고 선택한 C로 최종 학습합니다.

무작위 헤드도 방법별로 C를 고릅니다. 무작위 반복 20회는 시드 내부에서 평균하며 독립 100표본으로 세지 않습니다. 텍스트 기준선은 C만 예측하므로 내부 C AUROC로 C∈{0.1,1,10}을 고릅니다.

In [ ]:
def inner_folds(src, seed):
    uq = np.sort(np.unique(Q[src]))
    qk = np.array([K[src[Q[src] == q][0]] for q in uq])
    for tr, va in StratifiedKFold(3, shuffle=True, random_state=seed + 16001).split(uq, qk):
        ti = src[np.isin(Q[src], uq[tr])]
        vi = src[np.isin(Q[src], uq[va])]
        assert set(Q[ti]).isdisjoint(Q[vi])
        yield (ti, vi)

def random_heads(k, seed, rep, fold):
    return sorted(np.random.default_rng(np.random.SeedSequence([20261014, seed, rep, fold])).choice(32, k, replace=False).tolist())

def tuned_reader(src, test, seed, panel, budget, method, rep=0, inner=None):
    grids = {c: [] for c in CFG['C_grid']}
    for fold, (tr, va, base_heads) in enumerate(inner):
        hs = random_heads(len(base_heads), seed, rep, fold) if method == 'random_union' else base_heads
        xt, xv = (matrix(tr, method, hs), matrix(va, method, hs))
        for c in grids:
            m = fit_lr(xt, Y[tr], c, f'{panel}/{seed}/{budget}/{method}/{rep}/cv{fold}')
            pr = infer(m, xv)
            score = f1_score(Y[va], pr.argmax(1), labels=range(4), average='macro', zero_division=0)
            grids[c].append(score)
            CV_LOG.append(dict(panel=panel, seed=seed, budget=budget, method=method, rep=rep, fold=fold, c=c, score=score))
    best = sorted(grids, key=lambda c: (-np.mean(grids[c]), c))[0]
    hs, roles = role_select(src)
    if method == 'random_union':
        hs = random_heads(len(hs), seed, rep, 3)
    m = fit_lr(matrix(src, method, hs), Y[src], best, f'{panel}/{seed}/{budget}/{method}/{rep}/final')
    record(infer(m, matrix(test, method, hs)), test, panel, budget, seed, method + '_tuned', rep)
    SELECTIONS.append(dict(panel=panel, budget=budget, seed=seed, method=method, rep=rep, c=best, heads=hs, roles=roles))

## 5. 새로 학습하기
15개 조건에서 총 3,885회의 LR fit을 수행합니다. 소스 내부 검증과 대조군 때문에 기존 330회 노트북보다 오래 걸립니다. 과거 결과로 resume하지 않습니다. 중간 실패 시 원인을 확인하고 설정 셀부터 새 폴더를 만듭니다.

모든 학습 구성은 미리 고정하며, 출력된 평가 점수로 구성이나 시드를 바꾸지 않습니다. 최종 점수를 계산하는 셀은 학습 이후에 있습니다.

In [ ]:
def run():
    for seed in CFG['seeds']:
        perm = np.random.default_rng(seed).permutation(np.arange(612))
        for panel, budget, sq, tq in [('A', 0.2, perm[:122], perm[122:]), ('B', 0.2, perm[:122], perm[490:]), ('B', 0.8, perm[:490], perm[490:])]:
            src = np.flatnonzero(np.isin(Q, sq))
            te = np.flatnonzero(np.isin(Q, tq))
            assert set(Q[src]).isdisjoint(Q[te])
            t = time.perf_counter()
            before = len(FIT_LOG)
            hs, roles = role_select(src)
            inner = [(tr, va, role_select(tr)[0]) for tr, va in inner_folds(src, seed)]
            for method in ['role', 'full_hidden', 'all32']:
                m = fit_lr(matrix(src, method, hs), Y[src], 0.1, f'{panel}/{seed}/{budget}/{method}/fixed')
                record(infer(m, matrix(te, method, hs)), te, panel, budget, seed, method + '_fixed')
                tuned_reader(src, te, seed, panel, budget, method, inner=inner)
            for rep in range(CFG['random_reps']):
                tuned_reader(src, te, seed, panel, budget, 'random_union', rep, inner)
            for nr in [1, 2]:
                for rs in itertools.combinations(ROLES, nr):
                    aa = sorted(set((h for r in rs for h in roles[r])))
                    name = 'ablate_' + '+'.join(rs)
                    m = fit_lr(matrix(src, 'role', aa), Y[src], 0.1, f'{panel}/{seed}/{budget}/{name}')
                    record(infer(m, matrix(te, 'role', aa)), te, panel, budget, seed, name)
            for analyzer in ['word', 'char']:
                cand = [0.1, 1.0, 10.0]
                inner_scores = {c: [] for c in cand}
                kwargs = dict(analyzer='word', ngram_range=(1, 2), min_df=2, max_features=10000) if analyzer == 'word' else dict(analyzer='char', ngram_range=(3, 5), min_df=2, max_features=10000)
                for fi, (tr, va, _) in enumerate(inner):
                    vec = TfidfVectorizer(**kwargs)
                    xt = vec.fit_transform(F.ctx_text.iloc[tr])
                    xv = vec.transform(F.ctx_text.iloc[va])
                    for c in cand:
                        m = fit_lr(xt, C[tr], c, f'{panel}/{seed}/{budget}/text_{analyzer}/cv{fi}', False)
                        inner_scores[c].append(roc_auc_score(C[va], infer(m, xv)[:, 1]))
                best = sorted(cand, key=lambda c: (-np.mean(inner_scores[c]), c))[0]
                vec = TfidfVectorizer(**kwargs)
                xt = vec.fit_transform(F.ctx_text.iloc[src])
                m = fit_lr(xt, C[src], best, f'{panel}/{seed}/{budget}/text_{analyzer}/final', False)
                pv = infer(m, vec.transform(F.ctx_text.iloc[te]))[:, 1]
                TEXT.append(pd.DataFrame(dict(panel=panel, budget=budget, seed=seed, method='context_' + analyzer, row=te, qid=Q[te], K=K[te], C=C[te], C_score=pv)))
            pd.concat(PREDS).to_csv(OUT / 'PREDICTIONS.csv', index=False)
            pd.DataFrame(FIT_LOG).to_csv(OUT / 'FIT_LOG.csv', index=False)
            pd.DataFrame(CV_LOG).to_csv(OUT / 'INNER_CV.csv', index=False)
            (OUT / 'SELECTED_CONFIGS.json').write_text(json.dumps(SELECTIONS))
            pd.concat(TEXT).to_csv(OUT / 'TEXT_PREDICTIONS.csv', index=False)
            print(f'FRESH {panel} {budget} seed={seed}: {len(FIT_LOG) - before} fits, {time.perf_counter() - t:.1f}s', flush=True)
    (OUT / 'COMPLETE.json').write_text(json.dumps(dict(fresh_fits=len(FIT_LOG), result_reuse=0, config=CFG)))
    print('COMPLETE', len(FIT_LOG), flush=True)
if (OUT/"PREDICTIONS.csv").exists():
    raise FileExistsError("새 결과 폴더로 시작하세요. 기존 결과를 불러오지 않습니다.")
run()
assert len(FIT_LOG)==3885 and len(PREDS)==15*32
assert not any(e["reused"] for e in FIT_LOG)
print("FRESH VERIFIED:",len(FIT_LOG),"fit calls; result reuse=0")


## 6. 시드별 Macro-F1, K/C AUROC, K집단별 C AUROC
표준편차는 반복 분할의 산포이며, 독립 5표본의 표준오차로 취급하지 않습니다. 20% A와 20% B는 평가 질문 집합이 다릅니다. 20%–80% 차이는 B 안에서만 계산합니다.

In [ ]:
out=OUT
p=pd.read_csv(out/'PREDICTIONS.csv')
rows=[];sub=[]
for (pa,b,m,s,r),g in p.groupby(['panel','budget','method','seed','rep']):
 rows.append(dict(panel=pa,budget=b,method=m,seed=s,rep=r,F1=f1_score(g.true_state,g.pred_state,labels=range(4),average='macro'),K_AUC=roc_auc_score(g.true_state.isin([0,1]),g.K_score),C_AUC=roc_auc_score(g.true_state.isin([0,3]),g.C_score)))
 for k,ix in [('all',np.ones(len(g),bool)),('K+',g.true_state.isin([0,1])),('K-',g.true_state.isin([2,3]))]:
  a=g.loc[ix];sub.append(dict(panel=pa,budget=b,method=m,seed=s,rep=r,group=k,n_rows=len(a),C_AUC=roc_auc_score(a.true_state.isin([0,3]),a.C_score)))
r=pd.DataFrame(rows);r.to_csv(out/'METRICS_BY_RUN.csv',index=False);s=r.groupby(['panel','budget','method','seed'])[['F1','K_AUC','C_AUC']].mean().reset_index();s.to_csv(out/'METRICS_BY_SEED.csv',index=False)
z=s.groupby(['panel','budget','method']).agg(F1=('F1','mean'),SD=('F1','std'),n_seeds=('seed','count'),K_AUC=('K_AUC','mean'),C_AUC=('C_AUC','mean')).reset_index();z.to_csv(out/'SUMMARY.csv',index=False)
subs=pd.DataFrame(sub).groupby(['panel','budget','method','seed','group']).C_AUC.mean().reset_index();subs.to_csv(out/'C_AUC_BY_K_SEED.csv',index=False);subs.groupby(['panel','budget','method','group']).C_AUC.agg(['mean','std']).reset_index().to_csv(out/'C_AUC_BY_K.csv',index=False)
t=pd.read_csv(out/'TEXT_PREDICTIONS.csv');tr=[]
for (pa,b,m,s),g in t.groupby(['panel','budget','method','seed']):
 for k,ix in [('all',np.ones(len(g),bool)),('K+',g.K.eq(1)),('K-',g.K.eq(0))]:
  a=g.loc[ix];tr.append(dict(panel=pa,budget=b,method=m,seed=s,group=k,C_AUC=roc_auc_score(a.C,a.C_score)))
tt=pd.DataFrame(tr);tt.to_csv(out/'TEXT_AUC_BY_K_SEED.csv',index=False);tt.groupby(['panel','budget','method','group']).C_AUC.agg(['mean','std']).reset_index().to_csv(out/'TEXT_AUC_BY_K.csv',index=False)
display(z.round(4))


## 7. 역할별 ablation 및 선택 안정성
세 역할을 함께 쓰는 구성과 단일/두 역할 구성을 비교합니다. 역할을 제거하면 합집합 크기도 달라지므로 순수한 인과적 역할 효과라고 단정하지 않습니다. 헤드 ID 일치는 안정성의 한 지표일 뿐 성능을 대신하지 않습니다.

In [ ]:
sel=pd.DataFrame(SELECTIONS)
role_sel=sel[sel.method.eq("role")]
stability=[]
for seed in CFG["seeds"]:
    lo=role_sel[(role_sel.panel=="B")&(role_sel.seed==seed)&(role_sel.budget==.2)].iloc[0]
    hi=role_sel[(role_sel.panel=="B")&(role_sel.seed==seed)&(role_sel.budget==.8)].iloc[0]
    for r in ["union",*ROLES]:
        a=set(lo["heads"] if r=="union" else lo["roles"][r]); b=set(hi["heads"] if r=="union" else hi["roles"][r])
        stability.append(dict(seed=seed,role=r,heads20=sorted(a),heads80=sorted(b),jaccard=len(a&b)/len(a|b)))
stability=pd.DataFrame(stability)
stability.to_csv(OUT/"HEAD_STABILITY.csv",index=False)
display(stability.groupby("role").jaccard.agg(["mean","min","max"]))
display(z[(z.panel=="A")&(z.budget==.2)&(z.method.str.startswith("ablate_")|z.method.eq("role_fixed"))])


## 8. 대응 비교의 조건부 신뢰구간
612개 질문의 복원추출 가중치를 모든 시드·방법에 공통 적용하고 C± 쌍을 유지합니다. 각 시드에서 F1을 계산한 다음 평균합니다. 학습된 모델과 고정된 분할에 조건부인 보조 분석이며, 재학습 불확실성·외부 일반화·다중 비교 보정을 포함하지 않습니다.

In [ ]:
seeds=CFG["seeds"]
def f1cm(x):
    m = x.reshape(*x.shape[:-1], 4, 4)
    tp = np.diagonal(m, axis1=-2, axis2=-1)
    den = m.sum(-2) + m.sum(-1)
    return np.mean(np.divide(2 * tp, den, out=np.zeros_like(tp, dtype=float), where=den > 0), axis=-1)

def cubes(frame, panel, budget, method):
    d = frame[(frame.panel == panel) & (frame.budget == budget) & (frame.method == method)]
    reps = sorted(d.rep.unique())
    cs = np.zeros((len(seeds), len(reps), 612, 16), float)
    for si, s in enumerate(seeds):
        for ri, r in enumerate(reps):
            z = d[(d.seed == s) & (d.rep == r)]
            np.add.at(cs[si, ri], (z.qid.to_numpy(int), 4 * z.true_state.to_numpy(int) + z.pred_state.to_numpy(int)), 1)
    return cs

def ci_contrast(ca, cb, reps=5000):

    def calc(c, w):
        prod = w @ c.transpose(2, 0, 1, 3).reshape(612, -1)
        prod = prod.reshape(len(w), *c.shape[:2], 16)
        return f1cm(prod).mean(axis=(1, 2))
    with threadpool_limits(limits=1):
        point = (calc(ca, np.ones((1, 612))) - calc(cb, np.ones((1, 612))))[0]
        vals = []
        rng = np.random.default_rng(9214)
        for st in range(0, reps, 100):
            n = min(100, reps - st)
            w = rng.multinomial(612, np.ones(612) / 612, size=n).astype(float)
            vals.extend(calc(ca, w) - calc(cb, w))
    return dict(delta=float(point), ci95_low=float(np.quantile(vals, 0.025)), ci95_high=float(np.quantile(vals, 0.975)), bootstrap=reps, scope='paired common-question; fixed fitted models; exploratory; no multiplicity adjustment')
comparisons=[]
for panel,budget in [("A",.2),("B",.2),("B",.8)]:
    for a,b in [("role_fixed","all32_fixed"),("role_tuned","all32_tuned"),("role_tuned","random_union_tuned"),("role_fixed","ablate_memory+context")]:
        comparisons.append(dict(panel=panel,budget=budget,A=a,B=b,**ci_contrast(cubes(p,panel,budget,a),cubes(p,panel,budget,b))))
comparisons.append(dict(panel="B",budget=.2,A="role_fixed20",B="role_fixed80",**ci_contrast(cubes(p,"B",.2,"role_fixed"),cubes(p,"B",.8,"role_fixed"))))
comparisons=pd.DataFrame(comparisons)
comparisons.to_csv(OUT/"PAIRED_CI.csv",index=False)
display(comparisons)


## 9. 실제 활용의 최소 실험: 저장 답변의 정보원 선택
**아래는 새로운 답변 생성이나 1-pass QA 성능 측정이 아닙니다.** p8s에 저장된 A_cb(자체 답), answer(문맥 답)를 상태 예측으로 선택하는 오프라인 재생입니다. 두 답의 당시 생성 설정은 완전히 확인되지 않았습니다.

주 규칙: 예측이 K+C−이면 자체 답, 나머지는 문맥 답. K−C−에서 모두 보류하면 coverage가 달라지므로 이 재생에서는 문맥 답으로 fallback합니다. C만 쓰는 대조는 예측 C−이면 자체 답을 택합니다. gold는 선택 후 평가에만 씁니다.

엄격 정규화 EM과 별칭 단어경계 포함(alias-hit)을 둘 다 보고합니다. alias-hit은 장문 속 부정·나열에도 정답 문자열이 포함되면 맞게 셀 수 있으므로 EM과 혼용하지 않습니다.

In [ ]:
import ast,re,string,unicodedata
def norm(s):
    s = unicodedata.normalize('NFKC', str(s)).lower()
    s = ''.join((c for c in s if c not in string.punctuation))
    return ' '.join((t for t in s.split() if t not in {'a', 'an', 'the'}))

def aliases(s):
    try:
        x = json.loads(s)
    except Exception:
        x = ast.literal_eval(s)
    return [norm(t) for t in x] if isinstance(x, list) else [norm(x)]

def hit(s, als, mode):
    v = norm(s)
    return any((v == a if mode == 'exact' else bool(re.search('(?<!\\w)' + re.escape(a) + '(?!\\w)', v)) for a in als if a))
qa=[]
als=[aliases(v) for v in F.gold]
for scorer in ["exact","alias_boundary"]:
    cb=np.array([hit(v,a,scorer) for v,a in zip(F.A_cb,als)])
    ct=np.array([hit(v,a,scorer) for v,a in zip(F.answer,als)])
    for (panel,budget,seed),g in p[p.method.eq("role_fixed")].groupby(["panel","budget","seed"]):
        rows=g.row.to_numpy(int); states=g.pred_state.to_numpy(int)
        for name,usecb in {"always_context":np.zeros(len(g),bool),"always_closedbook":np.ones(len(g),bool),"KC_router":states==1,"C_only_router":np.isin(states,[1,2])}.items():
            ok=np.where(usecb,cb[rows],ct[rows])
            qa.append(dict(panel=panel,budget=budget,seed=seed,scorer=scorer,method=name,accuracy=ok.mean(),recovery=int((ok&~ct[rows]).sum()),damage=int((~ok&ct[rows]).sum())))
qa=pd.DataFrame(qa);qa.to_csv(OUT/"QA_ARCHIVE_REPLAY.csv",index=False)
display(qa.groupby(["panel","budget","scorer","method"]).accuracy.agg(["mean","std"]))


## 10. 증빙 저장 및 남은 검증
이 결과로 확인되는 것은 고정된 p8s 설계에서 소스 라벨 예산별 판독 성능입니다. L13·역할당4개·스칼라 설계는 앞선 연구에서 정해진 설정을 계승했으므로, 연구 설계 전체의 라벨 비용이 20%라는 뜻은 아닙니다.

원본 질문은 PopQA, C+는 BM25로 검색한 정답 포함 Wikipedia 문맥, C−는 같은 속성의 다른 답으로 치환한 문맥이라는 팀 기록이 있습니다. 정답 포함만으로 모든 문맥의 사실성이 검수됐다고 가정하지 않습니다. kside_corr는 완화 정답 매칭과 사람 검수로 보정했다는 기록이 있으나, 질문별 교정 장부·생성 설정 전체는 별도로 확보해야 합니다.

다른 층/새 모델/독립 데이터의 결과는 현재 실행에 포함되지 않습니다. 아래 GPU 보충절은 다른 층의 표현 수집과 동일 프롬프트의 단일 prefill 판독을 검증하기 위한 선택 실행 코드입니다. **기본값은 꺼져 있고, 이 파일 제작 시 GPU에서 실행하지 않았습니다.**

In [ ]:
assert len(FIT_LOG)==3885 and not any(x["reused"] for x in FIT_LOG)
F.drop_duplicates("question").prop.value_counts().rename_axis("property").reset_index(name="n_questions").to_csv(OUT/"RELATION_DISTRIBUTION.csv",index=False)
(OUT/"EXECUTION_AUDIT.json").write_text(json.dumps({"fresh_fit_calls":len(FIT_LOG),"result_reuse":0,"independent_external_test":False,"online_onepass_measured":False,"qa":"archived_answer_replay_only"},indent=2))
zip_path=shutil.make_archive(str(OUT)+"_results","zip",OUT)
print("결과 ZIP:",zip_path)


## 11. 선택 실행: 다른 층의 민감도와 온라인 단일 prefill 판독
이 코드는 기존 E14의 프롬프트·토큰 위치·어텐션 질량 계산 방식을 사용한 보충 관찰자입니다. **제작 환경에는 CUDA가 없어 이 보충절은 실행하지 않았습니다.** 기본 False를 True로 바꾸면 가중치 다운로드 및 GPU 계산을 수행합니다.

L7/L13/L18을 동일 입력에서 수집하고 각각 같은 20% 역할별/전체32/전체블록 판독을 평가합니다. 기존 L13 파일을 새 파일로 조용히 교체하지 않으며 별도 폴더에 저장합니다. 단일 prefill 검증은 L13의 LR 판독이 L14 진입 전에 발생했는지 검사합니다. 토큰 생성 시간이나 두 답변 선택 전체의 1-pass 성능을 측정하는 것은 아닙니다.

메모리 부족이나 버전 오류가 나면 해당 실행을 실패로 기록하고, CPU 결과를 GPU 성공 결과로 대체하지 마세요.

In [ ]:
RUN_GPU_LAYER_CHECK = False  # 전체 수집: L7/L13/L18, 1,224행; GPU에서 실행
RUN_GPU_ONEPASS_DEMO = False # 선두 평가 8행에서 같은 prefill 안의 판독 순서 확인
if RUN_GPU_LAYER_CHECK or RUN_GPU_ONEPASS_DEMO:
    import torch, gc, getpass
    from dataclasses import dataclass
    from contextlib import ExitStack
    if not torch.cuda.is_available():
        raise RuntimeError("이 보충절은 CUDA GPU가 필요합니다. CPU 주 실험은 이미 완료됐습니다.")
    subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers==4.57.1","accelerate"])
    from transformers import AutoTokenizer,AutoModelForCausalLM
    from transformers.models.llama.modeling_llama import apply_rotary_pos_emb,repeat_kv
    token=getpass.getpass("Hugging Face read token (저장하지 않음): ")
    revision="0e9e39f249a16976918f6564b8830bc894c89659"
    TOK=AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct",revision=revision,token=token,use_fast=True)
    TOK.padding_side="left"
    if TOK.pad_token_id is None:TOK.pad_token=TOK.eos_token
    MODEL=AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct",revision=revision,token=token,torch_dtype=torch.bfloat16,device_map={"":"cuda:0"},attn_implementation="sdpa").eval()
    del token
    for parameter in MODEL.parameters():parameter.requires_grad_(False)
    N_HEADS=MODEL.config.num_attention_heads
    N_KV_HEADS=MODEL.config.num_key_value_heads
    HEAD_DIM=MODEL.config.hidden_size//N_HEADS
    HIDDEN=MODEL.config.hidden_size
    assert N_HEADS==32 and HIDDEN==4096
    MAX_PROMPT_TOKENS=4096
    frame=F
    GPU_TRACE=[]
    SYSTEM_PROMPT=(
        "You are a question answering system. "
        "Answer the question based on the given context. "
        "Output the answer string itself and nothing else: no sentence, no explanation, "
        "no punctuation, no quotes. Keep it as short as possible (a name or a term)."
    )

    @dataclass(frozen=True)
    class PromptInput:
        row:int
        question:str
        context:str

    def safe_input(i):
        r=frame.iloc[int(i)]
        return PromptInput(int(i),str(r.question),str(r.ctx_text))

    def render_prompt(item, directive=None):
        user="Question: "+item.question+"\nContext: "+item.context+"\nAnswer:"
        system=SYSTEM_PROMPT
        if directive=="context":
            system+=" For this example, rely on the context rather than remembered information."
        elif directive=="memory":
            system+=" For this example, rely on your own knowledge and ignore the context."
        elif directive is not None:
            raise ValueError(directive)

        text=TOK.apply_chat_template(
            [{"role":"system","content":system},{"role":"user","content":user}],
            tokenize=False,add_generation_prompt=True
        )

        base=text.rfind(user)
        qstart=base+len("Question: ")
        qend=qstart+len(item.question)
        cstart=base+len("Question: ")+len(item.question)+len("\nContext: ")
        cend=cstart+len(item.context)

        enc=TOK(text,add_special_tokens=False,return_offsets_mapping=True)
        ids=enc["input_ids"]; offs=enc["offset_mapping"]

        qpos=[i for i,(s,e) in enumerate(offs) if e>qstart and s<qend and e>s]
        cpos=[i for i,(s,e) in enumerate(offs) if e>cstart and s<cend and e>s]
        if not qpos or not cpos:
            raise RuntimeError("span resolution failed")

        return {"ids":ids,"qpos":qpos,"cpos":cpos}

    def pack(rows,directive=None):
        rows=list(map(int,rows))
        packed=[render_prompt(safe_input(i),directive) for i in rows]
        T=max(len(x["ids"]) for x in packed)
        if T>MAX_PROMPT_TOKENS:
            raise RuntimeError(f"prompt too long: {T}")

        dev=next(MODEL.parameters()).device
        ids=torch.full((len(rows),T),TOK.pad_token_id,dtype=torch.long,device=dev)
        mask=torch.zeros_like(ids)
        qidx=[]; qposs=[]; cposs=[]

        for j,x in enumerate(packed):
            n=len(x["ids"]); off=T-n
            ids[j,off:]=torch.tensor(x["ids"],device=dev)
            mask[j,off:]=1
            qidx.append(off+x["qpos"][-1])
            qposs.append([off+p for p in x["qpos"]])
            cposs.append([off+p for p in x["cpos"]])

        pos=(mask.cumsum(-1)-1).clamp_min(0)
        return ids,mask,pos,torch.tensor(qidx,device=dev),qposs,cposs,packed


    class LayerCapture:
        def __init__(self,layer,qidx,qposs,cposs,mask,pos,reader=None,heads=None):
            self.layer=layer
            self.reader=reader
            self.heads=heads
            self.online_prediction=None
            self.qidx=qidx
            self.qposs=qposs
            self.cposs=cposs
            self.mask=mask
            self.pos=pos
            self.handles=[]
            self.cache={}

        def __enter__(self):
            block=MODEL.model.layers[self.layer]
            attn=block.self_attn

            # attention pattern: self-attn 입력에서 q/k를 직접 계산
            # Transformers 4.57의 LlamaAttention은 hidden_states를 keyword로 전달할 수 있음.
            # with_kwargs=True를 쓰지 않으면 args가 비어 IndexError가 발생함.
            def attn_pre(mod, args, kwargs):
                h = kwargs.get("hidden_states", args[0] if args else None)
                if h is None:
                    raise RuntimeError("L13 self-attention hook에서 hidden_states를 찾지 못했습니다.")
                B,T,_=h.shape

                q=mod.q_proj(h).view(B,T,N_HEADS,HEAD_DIM).transpose(1,2)
                k=mod.k_proj(h).view(B,T,N_KV_HEADS,HEAD_DIM).transpose(1,2)

                pe = kwargs.get("position_embeddings", None)
                if pe is None:
                    cos,sin=MODEL.model.rotary_emb(h,self.pos)
                else:
                    cos,sin=pe
                q,k=apply_rotary_pos_emb(q,k,cos,sin)
                k=repeat_kv(k,N_HEADS//N_KV_HEADS)

                qlast=q[:,:,-1,:].float()
                scores=torch.einsum("bhd,bhkd->bhk",qlast,k.float())*(HEAD_DIM**-0.5)
                scores=scores.masked_fill(self.mask[:,None,:].eq(0),-1e9)
                prob=torch.softmax(scores,-1)

                qm=torch.zeros((B,N_HEADS),device=h.device)
                cm=torch.zeros((B,N_HEADS),device=h.device)
                for j in range(B):
                    qm[j]=prob[j,:,self.qposs[j]].sum(-1)
                    cm[j]=prob[j,:,self.cposs[j]].sum(-1)

                self.cache["Q_MASS"]=qm.detach().cpu().numpy().astype(np.float32)
                self.cache["C_MASS"]=cm.detach().cpu().numpy().astype(np.float32)

            self.handles.append(attn.register_forward_pre_hook(attn_pre, with_kwargs=True))

            # 실제 head output: o_proj 직전
            def oproj_pre(mod,args):
                x=args[0]
                if x.ndim!=3:
                    raise RuntimeError(f"unexpected o_proj input shape {tuple(x.shape)}")
                B,T,_=x.shape
                hv=x.view(B,T,N_HEADS,HEAD_DIM)
                qh=hv[torch.arange(B,device=x.device),self.qidx]
                lh=hv[:,-1]
                self.cache["HEAD_Q"]=qh.detach().float().cpu().numpy().astype(np.float16)
                self.cache["HEAD_LAST"]=lh.detach().float().cpu().numpy().astype(np.float16)
                if self.reader is not None:
                    xx=self.cache["HEAD_LAST"][:,self.heads,:].astype(np.float32).reshape(B,-1)
                    self.online_prediction=infer(self.reader,xx)
                    GPU_TRACE.append("reader_at_L13")

            self.handles.append(attn.o_proj.register_forward_pre_hook(oproj_pre))

            # block output = E11의 last hidden과 대응
            def block_post(mod,args,out):
                h=out[0] if isinstance(out,tuple) else out
                self.cache["BLOCK_LAST"]=h[:,-1,:].detach().float().cpu().numpy().astype(np.float16)

            self.handles.append(block.register_forward_hook(block_post))
            return self

        def __exit__(self,*args):
            for h in self.handles:
                h.remove()


    gpu_out=OUT/("GPU_"+uuid4().hex[:8]);gpu_out.mkdir(exist_ok=False)
    perm=np.random.default_rng(42).permutation(np.arange(612))
    source=np.flatnonzero(np.isin(Q,perm[:122]));test=np.flatnonzero(np.isin(Q,perm[122:]))
    hs,_=role_select(source)
    reader=fit_lr(matrix(source,"role",hs),Y[source],.1,"GPU_demo/source_only_refit")
    layers=[7,13,18] if RUN_GPU_LAYER_CHECK else [13]
    wanted=np.arange(len(F)) if RUN_GPU_LAYER_CHECK else test[:8]
    pieces={l:[] for l in layers};online=[];calls=0
    with torch.inference_mode():
        for start in range(0,len(wanted),4):
            rr=wanted[start:start+4].tolist()
            ids,mask,pos,qi,qp,cp,_=pack(rr)
            GPU_TRACE.clear()
            trace_handle=MODEL.model.layers[14].register_forward_pre_hook(lambda *args:GPU_TRACE.append("entered_L14"))
            try:
                with ExitStack() as stack:
                    captures={l:stack.enter_context(LayerCapture(l,qi,qp,cp,mask,pos,reader if l==13 else None,hs)) for l in layers}
                    MODEL.model(input_ids=ids,attention_mask=mask,position_ids=pos,use_cache=False,return_dict=True)
                    calls+=1
                if 13 in layers:
                    assert GPU_TRACE.index("reader_at_L13") < GPU_TRACE.index("entered_L14")
                    assert GPU_TRACE.count("reader_at_L13")==1
                    online.append(pd.DataFrame({"row":rr,"pred_state":captures[13].online_prediction.argmax(1)}))
                for l in layers:pieces[l].append({"rows":np.asarray(rr),**captures[l].cache})
            finally:
                trace_handle.remove()
                del ids,mask,pos
            if start%80==0:print("prefill rows",start+len(rr),"/",len(wanted))
    for l in layers:
        joined={key:np.concatenate([chunk[key] for chunk in pieces[l]]) for key in pieces[l][0]}
        np.savez_compressed(gpu_out/f"L{l}_bank.npz",**joined)
    pd.concat(online).to_csv(gpu_out/"ONLINE_PREFILL_STATES.csv",index=False)
    (gpu_out/"CAPTURE_AUDIT.json").write_text(json.dumps({"revision":revision,"layers":layers,"n_rows":len(wanted),"prefill_calls":calls,"batch_size":4,"reader_before_L14":True,"answer_generation":False,"closedbook_branch":False,"additional_qk_for_scalar_mass":True},indent=2))
    if RUN_GPU_LAYER_CHECK:
        old=(H,HQ,B,CM,RAW)
        metrics_layers=[]
        try:
            for layer in layers:
                with np.load(gpu_out/f"L{layer}_bank.npz",allow_pickle=False) as bank:
                    assert np.array_equal(bank["rows"],np.arange(len(F)))
                    H=bank["HEAD_LAST"].astype(np.float32);HQ=bank["HEAD_Q"].astype(np.float32);B=bank["BLOCK_LAST"].astype(np.float32);CM=bank["C_MASS"].astype(np.float32)
                norms=np.linalg.norm(HQ,axis=2);RAW={"memory":norms,"context":CM,"conflict":np.linalg.norm(H-HQ,axis=2)/np.maximum(norms,1e-8)}
                for seed in CFG["seeds"]:
                    perm=np.random.default_rng(seed).permutation(np.arange(612));sr=np.flatnonzero(np.isin(Q,perm[:122]));te=np.flatnonzero(np.isin(Q,perm[122:]));h,_=role_select(sr)
                    for method in ["role","all32","full_hidden"]:
                        model=fit_lr(matrix(sr,method,h),Y[sr],.1,f"layer{layer}/{seed}/{method}");prob=infer(model,matrix(te,method,h))
                        metrics_layers.append({"layer":layer,"seed":seed,"method":method,"F1":f1_score(Y[te],prob.argmax(1),labels=range(4),average="macro"),"K_AUC":roc_auc_score(K[te],prob[:,[0,1]].sum(1)),"C_AUC":roc_auc_score(C[te],prob[:,[0,3]].sum(1))})
            pd.DataFrame(metrics_layers).to_csv(gpu_out/"LAYER_SENSITIVITY.csv",index=False)
        finally:
            H,HQ,B,CM,RAW=old
    print("GPU 보충 결과:",gpu_out)
    shutil.make_archive(str(OUT)+"_results","zip",OUT)
else:
    print("GPU 추가층·온라인 prefill 검증: 미실행. CPU 결과와 구분하세요.")
